<a href="https://colab.research.google.com/github/RyanMota-Dev/Api_Users_CICD/blob/main/pennylane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classificador Quântico Variacional (VQC) para Match de Voluntários
**Integrantes:** Luis Fernando, Ryan Mota e Douglas Barbosa
**Projeto Relacionado:** ABC do Bem (TCC)

## O Problema: Quem devemos recomendar?
Em nossa plataforma, precisamos decidir se uma vaga de ONG deve ser recomendada ou não para um voluntário.
Utilizamos duas variáveis principais (normalizadas entre 0 e pi):
1. **Afinidade de Habilidades:** Quantas competências o voluntário tem em comum com a vaga.
2. **Proximidade Geográfica:** Quão perto o voluntário mora do local do evento.

## O Circuito (2 Qubits + Emaranhamento)
Construímos um Classificador Quântico Variacional onde:
* O **Qubit 0** e o **Qubit 1** recebem as "features" (Habilidades e Distância) via *Quantum Embedding*.
* Aplicamos **Emaranhamento (CNOT)** para cruzar essas duas variáveis, pois uma pessoa muito habilidosa que mora muito longe pode não ser um bom match, assim como alguém que mora perto mas não tem as habilidades exigidas.
* O otimizador ajusta os pesos (rotações) ao longo de 15 épocas para "aprender" o padrão de um bom voluntário.

In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 97.2 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [ ]:
import pennylane as qml
import pennylane.numpy as np

# 1. Gerar dados simulados de Voluntários vs ONGs (normalizados entre 0 e pi)
# [Afinidade_Habilidades, Proximidade_Geografica]
np.random.seed(42)
X_dados = np.array([
    [0.8, 0.9], # Boas habilidades, mora muito perto
    [0.9, 0.7], # Ótimas habilidades, mora razoavelmente perto
    [0.7, 0.8], # Boas habilidades, mora perto
    [0.1, 0.2], # Não sabe fazer a função, mora muito longe
    [0.9, 0.1], # Ótimas habilidades, mas mora extremamente longe (Inviável)
    [0.2, 0.8]  # Mora do lado, mas não tem as habilidades necessárias
], requires_grad=False)

# Rótulos (Y): +1 para MATCH (Recomendado), -1 para NO-MATCH (Não recomendado)
Y_rotulos = np.array([1, 1, 1, -1, -1, -1], requires_grad=False)

# 2. Configurar o Dispositivo de 2 Qubits
dev = qml.device("default.qubit", wires=2)

# 3. Construir o VQC (Classificador Quântico Variacional - ABC do Bem)
@qml.qnode(dev)
def vqc_circuito_match(pesos_treinaveis, x_features):
    # A) Quantum Embedding: Inserindo Habilidades e Distância nos Qubits
    qml.RX(x_features[0], wires=0)
    qml.RY(x_features[1], wires=1)

    # B) Camada Variacional (Pesos ajustados pela Inteligência Artificial)
    qml.Rot(*pesos_treinaveis[0], wires=0)
    qml.Rot(*pesos_treinaveis[1], wires=1)

    # C) Emaranhamento para cruzar as variáveis (Habilidades x Distância)
    qml.CNOT(wires=[0, 1])

    # D) Segunda camada de rotação ajustável
    qml.Rot(*pesos_treinaveis[2], wires=0)
    qml.Rot(*pesos_treinaveis[3], wires=1)

    # Retorna a expectativa para decidir a classe (+1 ou -1)
    return qml.expval(qml.PauliZ(1))

# 4. Definir a função de Custo (usando qml.numpy para aceitar os ArrayBoxes do professor)
def custo_classificador(pesos, X, Y):
    perdas = [ (vqc_circuito_match(pesos, x) - y) ** 2 for x, y in zip(X, Y) ]
    return qml.math.mean(qml.math.stack(perdas))

# 5. Inicializar pesos aleatórios (4 conjuntos de 3 ângulos, como o professor fez)
pesos_iniciais = np.random.random((4, 3), requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.4)

# 6. Treinamento do Modelo Quântico
pesos_otimizados = pesos_iniciais
print("=== INICIANDO O TREINAMENTO DO VQC (ABC DO BEM) ===")
for epoca in range(15):
    pesos_otimizados, custo_val = opt.step_and_cost(lambda p: custo_classificador(p, X_dados, Y_rotulos), pesos_otimizados)
    if epoca % 3 == 0 or epoca == 14:
        print(f"Época {epoca:2d} | Custo (Erro): {custo_val:.4f}")

# 7. Testando o modelo treinado com uma NOVA candidatura (Um voluntário que acabou de se cadastrar)
# Exemplo: Voluntário com excelente compatibilidade de habilidades (0.85) e que mora perto (0.75)
novo_voluntario = np.array([0.85, 0.75], requires_grad=False)
resultado = vqc_circuito_match(pesos_otimizados, novo_voluntario)
classe_prevista = "MATCH! (Recomendar Vaga)" if resultado > 0 else "NÃO RECOMENDADO"

print(f"\nResultado para nova candidatura (Habilidades/Distância 0.85, 0.75): Saída = {resultado:.4f} -> {classe_prevista}")


=== INICIANDO O TREINAMENTO DO VQC (ABC DO BEM) ===
Época  0 | Custo (Erro): 1.3427
Época  3 | Custo (Erro): 1.1097
Época  6 | Custo (Erro): 0.8836
Época  9 | Custo (Erro): 0.7719
Época 12 | Custo (Erro): 0.7235
Época 14 | Custo (Erro): 0.7056

Resultado para nova candidatura (Habilidades/Distância 0.85, 0.75): Saída = 0.1895 -> MATCH! (Recomendar Vaga)
